In [120]:
# RUN THIS CELL FIRST
import os
import math
from collections import OrderedDict

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import v2

import numpy as np
from numpy import allclose, isclose

from collections.abc import Callable
from sklearn.model_selection import train_test_split

ASSETS_PATH = "data/assets"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
transformations = transforms.Compose(
    [
        v2.Resize((128,128)),
        v2.RandomHorizontalFlip(),
        # v2.ColorJitter(brightness=(0.1,0.3), contrast=(0.1,0.3)),
        v2.RandomGrayscale(p=0.3),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        # transforms.RandomInvert(p=0.3),
        v2.Normalize(mean=[0.5, 0.5, 0.5],
                                 std=[0.5, 0.5, 0.5])
    ]
)


cuda


In [121]:
dataset = datasets.ImageFolder(ASSETS_PATH, transform=transformations)
class_labels = list(dataset.class_to_idx.keys())
indices = list(range(len(dataset)))
labels = [y for _, y in dataset]
train_idx, test_idx = train_test_split(indices, test_size=0.2, stratify=labels)

train_set = Subset(dataset, train_idx)
test_set = Subset(dataset, test_idx)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, drop_last=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=True)

print(labels)



[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 13, 13, 1

In [122]:
def train_model(model: nn.Module, dataloader: DataLoader, epochs: int = 20):
    """
    Trains the model for a specified number of epochs/iterations
    
    Parameters
    ---------- 
        model: A PyTorch model to be trained
        dataloader : A DataLoader object that provides batches of the training data
        epochs  : Number of epochs, default of 20
        
    Returns
    -------
        The final model and the loss curve (per epoch)
    """

    losses = []
    loss_fn = nn.CrossEntropyLoss()
    # Set model to training mode. 
    # See (https://stackoverflow.com/questions/60018578/what-does-model-eval-do-in-pytorch) if curious.
    model.train() 
    optimiser = torch.optim.SGD(model.parameters(), momentum=0.9, lr=0.01)
    # optimiser = torch.optim.AdamW(model.parameters())
    for i in range(epochs):
        epoch_loss = 0.0
        for x_batch, y_batch in dataloader:
            optimiser.zero_grad()
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            y_output = model.predict_proba(x_batch)
            loss = loss_fn(y_output, y_batch)
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item()
            
        print(f"Epoch {i+1}/{epochs}, Loss: {epoch_loss:.4f}")
        losses.append(epoch_loss)

    return model, losses

In [123]:
class TestCNN(nn.Module):
    def __init__(self, classes: int):
        super().__init__()
        self.conv = nn.Sequential(
                        nn.Conv2d(3, 32, (3,3)),
                        nn.Dropout(0.5),
                        nn.MaxPool2d((2,2)),
                        nn.LeakyReLU(0.1),
                        nn.Conv2d(32, 64, (3,3)),
                        nn.Dropout(0.5),
                        nn.MaxPool2d((2,2)),
                        nn.LeakyReLU(0.1),
                        nn.Conv2d(64, 128, (3,3)),
                        nn.Dropout(0.5),
                        nn.MaxPool2d((2,2)),
                        nn.LeakyReLU(0.1),
                        nn.Conv2d(128, 256, (3,3)),
                        nn.Dropout(0.5),
                        nn.MaxPool2d((2,2)),
                        nn.LeakyReLU(0.1),
                        nn.Conv2d(256, 512, (3,3)),
                        nn.Dropout(0.5),
                        nn.MaxPool2d((2,2)),
                        nn.LeakyReLU(0.1),
                    )

        self.fc = nn.Sequential(
                        nn.Linear(512, 1024),
                        nn.LeakyReLU(0.1),
                        nn.Linear(1024, 512),
                        nn.LeakyReLU(0.1),
                        nn.Linear(512, 256),
                        nn.LeakyReLU(0.1),
                        nn.Linear(256, 128),
                        nn.LeakyReLU(0.1),
                        nn.Linear(128, classes),
                    )
        self.gap = nn.AdaptiveAvgPool2d(1)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """ YOUR CODE HERE """
        x = self.conv(x)
        """ YOUR CODE END HERE """
        x = self.gap(x) # GAP – do not remove this line
        """ YOUR CODE HERE """
        x = x.view(x.shape[0], -1)
        out = self.fc(x)
        """ YOUR CODE END HERE """
        return out
    
    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        out = self.forward(x)
        return torch.softmax(out, dim = 1)

In [124]:
test_model, test_losses = train_model(TestCNN(14).to(device), train_loader, epochs = 200)

Epoch 1/200, Loss: 10.5560
Epoch 2/200, Loss: 10.5562
Epoch 3/200, Loss: 10.5560
Epoch 4/200, Loss: 10.5562
Epoch 5/200, Loss: 10.5563
Epoch 6/200, Loss: 10.5559
Epoch 7/200, Loss: 10.5560
Epoch 8/200, Loss: 10.5559
Epoch 9/200, Loss: 10.5560
Epoch 10/200, Loss: 10.5558
Epoch 11/200, Loss: 10.5558
Epoch 12/200, Loss: 10.5559
Epoch 13/200, Loss: 10.5557
Epoch 14/200, Loss: 10.5559
Epoch 15/200, Loss: 10.5556
Epoch 16/200, Loss: 10.5558
Epoch 17/200, Loss: 10.5556
Epoch 18/200, Loss: 10.5556
Epoch 19/200, Loss: 10.5556
Epoch 20/200, Loss: 10.5553
Epoch 21/200, Loss: 10.5555
Epoch 22/200, Loss: 10.5556
Epoch 23/200, Loss: 10.5558
Epoch 24/200, Loss: 10.5552
Epoch 25/200, Loss: 10.5555
Epoch 26/200, Loss: 10.5553
Epoch 27/200, Loss: 10.5553
Epoch 28/200, Loss: 10.5554
Epoch 29/200, Loss: 10.5553
Epoch 30/200, Loss: 10.5551
Epoch 31/200, Loss: 10.5551
Epoch 32/200, Loss: 10.5554
Epoch 33/200, Loss: 10.5550
Epoch 34/200, Loss: 10.5550
Epoch 35/200, Loss: 10.5551
Epoch 36/200, Loss: 10.5552
E

In [125]:
def get_accuracy(scores: torch.Tensor, labels: torch.Tensor) -> int | float:
    _, predictions = torch.max(scores, 1)
    
    # Per-class accuracy breakdown
    classes = torch.unique(labels)
    for cls in classes:
        mask = labels == cls
        cls_acc = (predictions[mask] == labels[mask]).float().mean().item()
        print(f"Class {class_labels[cls.item()]}: {cls_acc:.2%}")
    
    overall = (predictions == labels).float().mean().item()
    print(f"Overall: {overall:.2%}")
    return overall

In [114]:
print(classes)

['boots', 'box', 'coin', 'exit', 'floor', 'gem', 'ghost', 'human', 'key', 'lava', 'locked', 'opened', 'shield', 'wall']


In [133]:
with torch.no_grad():
    test_model.eval()
    for i, data in enumerate(test_loader):
        x, y = data
        x, y = x.to(device), y.to(device)
        pred = test_model.predict_proba(x)
        acc = get_accuracy(pred, y)
        print(f"test accuracy: {acc}")

Class boots: 0.00%
Class box: 0.00%
Class coin: 80.00%
Class exit: 0.00%
Class floor: 100.00%
Class gem: 0.00%
Class ghost: 0.00%
Class human: 100.00%
Class key: 60.00%
Class lava: 40.00%
Class locked: 0.00%
Class opened: 100.00%
Class shield: 0.00%
Class wall: 100.00%
Overall: 46.88%
test accuracy: 0.46875
Class boots: 0.00%
Class floor: 100.00%
Class gem: 0.00%
Overall: 33.33%
test accuracy: 0.3333333432674408
